# 03 – Full universe: historical S&P 500 members

Builds the backtest dataset for **every company that was in the S&P 500 at any month end since `backtest.start`**
(not only today's members):

1. **Point-in-time membership** – daily index composition since 1996 ([fja05680/sp500](https://github.com/fja05680/sp500), MIT) → month ends
2. **Ticker → CIK** – Wikipedia (current members) + SEC ticker list + manual rename table in `config.yaml` (FB → META…)
3. **Fundamentals** for ~600 companies (SEC XBRL, same pipeline as notebook 01)
4. **Monthly prices** (total return + split-adjusted close) and **split history**
5. **Piotroski F-score** and **Monte Carlo DCF values** per fiscal year

⏱️ First run: ~20–30 min (≈600 SEC downloads ~300 MB gzipped, ≈600 Yahoo split requests). Everything is cached.

⚠️ **Survivorship gap.** Acquired or bankrupt companies have no free price data (Yahoo deletes them) and no current
ticker. They are missing from the backtest. The coverage table below shows how big the gap is per year.

In [ ]:
from IPython.display import display
import pandas as pd

from stockpred.config import load_settings
from stockpred.edgar import EdgarClient
from stockpred.constituents import fetch_history, build_membership
from stockpred.fundamentals import build_panel
from stockpred.prices import split_history, monthly_prices
from stockpred.factors import add_piotroski, dcf_values

settings = load_settings()
P = settings.processed_dir
client = EdgarClient(settings.sec_user_agent, settings.raw_dir / "edgar",
                     settings.max_requests_per_second, settings.max_age_days)

## 1. Membership and survivorship coverage

In [ ]:
hist = fetch_history(settings.raw_dir / "sp500_history.csv")
wiki = pd.read_csv(settings.raw_dir / "sp500_constituents.csv")
membership, coverage = build_membership(client, hist, wiki, settings.ticker_renames, settings.backtest_start)
membership.to_parquet(P / "membership.parquet", index=False)

companies = membership.drop_duplicates("cik")[["ticker", "cik"]].drop_duplicates("ticker")
print(f"History up to {hist['date'].max():%Y-%m-%d}; {len(companies)} companies with a CIK")
display(coverage.T.style.format("{:.0%}"))

## 2. Fundamentals, splits, prices

In [ ]:
splits = split_history(companies["ticker"].tolist(), settings.raw_dir / "splits_full.parquet")
panel = build_panel(client, companies, start_year=settings.start_year, splits=splits,
                    predecessors=settings.cik_predecessors)
panel = add_piotroski(panel)
panel.to_parquet(P / "fundamentals_full.parquet", index=False)
print(panel.shape, panel["ticker"].nunique(), "companies")

In [ ]:
prices = monthly_prices(sorted(set(companies["ticker"]) | {"SPY"}), "2009-12-01", settings.raw_dir / "prices")
tri, close = prices
tri.to_parquet(P / "prices_tri_monthly.parquet")
close.to_parquet(P / "prices_close_monthly.parquet")
no_data = [t for t in tri.columns if tri[t].isna().all()]
print(f"{tri.shape[1]} tickers, {tri.index.min():%Y-%m} – {tri.index.max():%Y-%m}; no price data: {no_data}")

## 3. Monte Carlo DCF value per fiscal year

For every company and fiscal year, the DCF uses **only the history up to that year** (2,000 paths, median value).
Financials (by today's GICS sub-industry) are skipped. The value does not depend on the price, so it is computed
once per 10-K and compared with each later month-end price in notebook 05.

In [ ]:
sub_industry = dict(zip(wiki["ticker"].str.replace(".", "-"), wiki["sub_industry"]))
dcf = dcf_values(panel, sub_industry, n_sims=2_000)
dcf.to_parquet(P / "dcf_values_full.parquet", index=False)
print(len(dcf), "company-years,", dcf["ticker"].nunique(), "companies")